In [5]:
# tous les bib utilise|
# from transformers import pipeline
# from langdetect import detect, DetectorFactory
# import re 
import pandas

detection de langauge 

In [ ]:
# installer le modele localement
# ....
# print("Modèle enregistré avec succès dans le dossier ./local_models/lang_detector' ")

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_path = "./local_models/lang_detector/model.safetensors"

tokenizer = AutoTokenizer.from_pretrained("./local_models/lang_detector")
model = AutoModelForSequenceClassification.from_pretrained(model_path)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './local_models/lang_detector'. Use `repo_type` argument if needed.

In [ ]:
print(classifier("Bonjour tout le monde"))
print(classifier("السلام عليكم"))
print(classifier("wach rak labas kho"))  # darja

In [ ]:
df = pandas.read_csv('DATA\MergedComplaints.csv')
df["language_detecter"] = df['Commentaire client'].apply(detect_language_fr_ar_derja)

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_12720\266962571.py:1: SyntaxWarning: invalid escape sequence '\M'
  df = pandas.read_csv('DATA\MergedComplaints.csv')
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
df.drop(columns=['Defined Class','Classe associé'] , inplace=True)
df.to_csv("DATA/df_tempo.csv")

In [7]:
df = pandas.read_csv('DATA\df_tempo.csv')
df[ (df["language_detecter"] == 'Derja') & (df["Predicted class"] == "Problème / Réclamation") ].to_csv("DATA/voirDarja.csv")
df = pandas.read_csv('DATA/voirDarja.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_6596\1887294772.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pandas.read_csv('DATA\df_tempo.csv')


In [ ]:
#df_filtre = df[df['Commentaire client'].str.contains(r'\b3ndna\b', regex=True, na=False)]

#df_filtre = df[df['Commentaire client'].str.contains(r'(?i)(?<!\w)ça fait(?!\w)', regex=True, na=False)]

#df_filtre = df[df['Commentaire client'].str.contains(r'(^|\s)من(\s|$)', regex=True, na=False)]

# On met directement les plages de chiffres arabes et persans
filtre_chiffres = df[df['Commentaire client'].str.contains(r'[٠-٩۰-۹]', regex=True, na=False)]


# Supprime la limite d'affichage des lignes
pandas.set_option('display.max_rows', None)

# Affiche votre dataframe filtré
print(filtre_chiffres)

In [ ]:
df["Commentaire client"].loc[]

detecter le temp d'attente 

In [4]:

# comment savoir que la date extract est une date d'attente : d'apres le data la class reclamation ne contient que des dates d'attente 
# apres avoir detecter la langue si fr / ar --->


# example en arabe (same for french) cole et non cole
# 2 ans ,منذ شهر ,  عامين ,  5 ايام,  4 اشهر ,البارح , يومين , شهرين , عام , من البارح , 16 يوم , المساء , اسبوع , الايام , باليوم 9 , J + 24 ,  48 ساعة , ça fait 2 jours , اكثر من شهر
# example en darja 
# 3ndi 2 mois ,  3ndna 3ans w 7 chhr , rah 2mois,  7 chhor , 3am , سمانة , man lkhmis , مالبارح , من بارح  , بارحة,m lberh ,lberh, السمانة , w 2 ans , عندو 6اشهر 

# IMPORTANT POINT :
# pour check tout ca avec les patterens on doit negligier espace avant et apres
# il existe des chiffres en arabe 

# convertire chiffre arabe en fr 
def arNumbers_toFr(text:str):
    arabic_nums = "٠١٢٣٤٥٦٧٨٩"
    western_nums = "0123456789"
    trans = str.maketrans(arabic_nums, western_nums)
    return text.translate(trans)

# extract a number from a specifique pattern
def extractNum_fromPat(text, patterns):
    for p in patterns:
        match = re.search(p, text)
        if match:
            # pour eviter erruer 
            try:
                return int(match.group(1))
            except (IndexError, ValueError):
                continue 
    return None

darja_context = [
        r"\b(3ndi|3ndna|rah|men|man|ml|m|w|3and)\b"
    ]

def extractNum_fromContext(text,context):
    ...

def detect_waiting_time(text_cleaned) -> str : 
    
    text_cleaned = arNumbers_toFr()
    text_cleaned.lower()

    lang = detect_language_fr_ar_derja(text_cleaned)

    # definir les patterns 
    patterns_fr = [
        r"\b(\d+)\s*(?:jour|jours)\b",
        r"\b(\d+)\s*(?:mois)\b",
        r"\b(\d+)\s*(?:an|ans)\b",
        r"\b(\d+)\s*(?:heure|heures)\b",
        r"\b(\d+)\s*(?:semaine|semaines)\b",
        r"\bj\+?\s*(\d+)\b"
    ]

    patterns_ar = [
        r"\b\d+\s*(يوم|ايام)\b",
        r"\b\d+\s*(شهر|اشهر)\b",
        r"\b\d+\s*(عام|اعوام|سنوات)\b",
        r"\b\d+\s*(ساعة|ساعات)\b",
        r"\b\d+\s*(اسبوع|اسابيع)\b",
    ]

    darja_context = [
        r"\b(3ndi|3ndna|rah|men|man|ml|m|w|3and)\b"
    ]

    darja_duration = [
        r"\b\d+\s*(mois|chhr|chhor)\b",
        r"\b\d+\s*(ans|3am)\b",
        r"\b\d+\s*(jours|nhar|يوم)\b",
        r"\b\d+\s*(سمانة)\b",
        r"\b(lberh|m\s*lberh|barh|malbarh)\b"
    ]

    
    # logique 
    if lang == "fr":
        return extractNum_fromPat(text_cleaned , patterns_fr)

    elif lang == "ar":
        return extractNum_fromPat(text_cleaned, patterns_ar)

    elif lang == "darja":
        # d'abbord apllquer fr et ar
        tempo = extractNum_fromPat(text_cleaned , patterns_fr)
        if(tempo != None) : return extractNum_fromPat(text_cleaned, patterns_fr) 
        tempo = extractNum_fromPat(text_cleaned , patterns_ar)
        if(tempo != None) : return extractNum_fromPat(text_cleaned, patterns_ar) 

        # si context existe alors extract dure 
        has_context = any(re.search(p, text_cleaned) for p in darja_context)
        if(has_context) : 
            take_duration = any(re.search(p, text_cleaned) for p in darja_duration)
            return take_duration
        return None

    return None



EXTRCAT ENTITIES